In [ ]:
import torch
import sys
import numpy as np
sys.path.append('/data/chenyihao/LucaVirusTasks_source')
sys.path.append('/data/chenyihao/LucaVirusTasks_source/src/lucaquadruple/models')
from torch.utils.data import Dataset, DataLoader

In [2]:
import torch.nn.functional as F
from utilities import load_embedding

class LucaDataset(Dataset):
    def __init__(self, DataFrame):
        self.emb_file_name_a = ('matrix_' + DataFrame['seq_id_a']).tolist()
        self.emb_file_name_b = ('matrix_' + DataFrame['seq_id_b']).tolist()
        self.emb_file_name_c = ('matrix_' + DataFrame['seq_id_c']).tolist()
        self.emb_file_name_d = ('matrix_' + DataFrame['seq_id_d']).tolist()

        self.strainPassCats = convert_Pass2tensor(('<cls>' + DataFrame['serumPassCat'] + '<eos>' + DataFrame['virusPassCat'] + '<eos>').tolist())

        self.labels = torch.tensor(DataFrame['label'].tolist())
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        return self.emb_file_name_a[idx], self.emb_file_name_b[idx], self.emb_file_name_c[idx], self.emb_file_name_d[idx], \
               self.strainPassCats[idx], self.labels[idx]

def convert_Pass2tensor(pass_cats):
    result = [
        item.replace('<cls>', '0').replace('<eos>', '1').replace('<EGG>', '2').replace('<CELL>', '3').replace('<BOTH>', '4')
        for item in pass_cats
    ]
    result = torch.tensor([[int(number) for number in [char for char in item]] for item in result])
    return result

def list2df(mylist, period):
    merged_rows = []
    for i in range(0, len(mylist), period):
        merged_row = []
        for j in range(period):
            merged_row += mylist[i + j]
        merged_rows.append(merged_row)

    return pd.DataFrame(merged_rows)

def generate_matrix(matrix_list):
    seq_len = [mat.shape[0] for mat in matrix_list]
    max_len = max(seq_len)
    mask_list = []
    for i in range(len(matrix_list)): 
        matrix_list[i] = F.pad(matrix_list[i], (0, 0, 0, max_len - seq_len[i]))
        mask = torch.concat((torch.ones(1,seq_len[i]),torch.zeros(1,max_len-seq_len[i])),axis=1)
        mask_list.append(mask)
    matrix = torch.stack(matrix_list)
    mask = torch.stack(mask_list).view(len(matrix_list),max_len)
    return matrix, mask


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split

device = torch.device('cuda:0')
group_columns = ['seq_a', 'seq_b', 'seq_c', 'seq_d', 'serumPassCat', 'virusPassCat']
new_columns = ['seq_id_a', 'seq_id_b', 'seq_id_c', 'seq_id_d','seq_a', 'seq_b', 'seq_c', 'seq_d', 
               'serumPassCat', 'virusPassCat', 'serumName', 'virusName', 'label']
data_path = '/data/chenyihao/dataset'

Crick_all = pd.read_csv(data_path + '/all.csv')
dataframe = Crick_all.groupby(group_columns).agg({'seq_id_a': 'first', 'seq_id_b': 'first', 'seq_id_c': 'first', 'seq_id_d': 'first',
                                                  'seq_type_a': 'first', 'seq_type_b': 'first', 'seq_type_c': 'first', 'seq_type_d': 'first',
                                                  'serumName': 'first', 'virusName': 'first', 'label': 'mean'}).reset_index()
Crick_all_final = dataframe[new_columns]

Crick_H1N1 = pd.read_excel('../../../data/raw/data4model(Crick-H1N1).xlsx')
Crick_H3N2 = pd.read_excel('../../../data/raw/data4model(Crick-H3N2).xlsx')
Crick_serumType = pd.concat([Crick_H1N1,Crick_H3N2])[['virusName', 'serumType']].drop_duplicates(subset=['virusName']).reset_index(drop=True)
Crick_all_final = Crick_all_final.merge(right=Crick_serumType,how='left', left_on='virusName', right_on='virusName')

train_data, test_data = train_test_split(Crick_all_final, test_size=0.1, random_state=42)
train_data, valid_data = train_test_split(train_data, test_size=1/9, random_state=42)

In [4]:
test_data_H1N1 = test_data[test_data['serumType'] == 'H1N1']
test_data_H3N2 = test_data[test_data['serumType'] == 'H3N2']

H1N1_dataset = LucaDataset(test_data_H1N1)
H3N2_dataset = LucaDataset(test_data_H3N2)
test_dataset = LucaDataset(test_data)

H1N1_dataloader = DataLoader(H1N1_dataset, batch_size=80, shuffle=False)
H3N2_dataloader = DataLoader(H3N2_dataset, batch_size=80, shuffle=False)
Crick_test_dataloader = DataLoader(test_dataset, batch_size=80, shuffle=False)

device = torch.device("cuda:1")
model = torch.load('/data/chenyihao/fluProfiler/model/001.pth', weights_only=False, map_location=device)

In [5]:
import os
from tqdm import tqdm

# load embedding
sequence_names = pd.concat([test_data['seq_id_a'],test_data['seq_id_b'],
                            test_data['seq_id_c'],test_data['seq_id_d']]).unique().tolist()
sequence_names = ['matrix_' + item + '.pt' for item in sequence_names]
IDs, embeddings = load_embedding("/data/chenyihao/embedding", files=sequence_names)
emb_dict = dict(zip(IDs, embeddings))

Loading tensor: 100%|██████████| 6419/6419 [15:50<00:00,  6.75file/s]


In [9]:
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr

# 计算 R^2
r2_score


test_prediction_ls = []
test_reference_ls = []
test_logits_ls = []
model.eval()
for batch in H1N1_dataloader:
    with torch.no_grad():
        emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch
        
        matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
        matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
        matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
        matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])
        
        matrixs_a = matrixs_a.to(device)
        matrixs_b = matrixs_b.to(device)
        matrixs_c = matrixs_c.to(device)
        matrixs_d = matrixs_d.to(device)

        masks_a = masks_a.to(device)
        masks_b = masks_b.to(device)
        masks_c = masks_c.to(device)
        masks_d = masks_d.to(device)
        
        strainPassCats = strainPassCats.to(device)
        
        labels = labels.to(device)
        
        with torch.no_grad():
            loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c, matrices_d=matrixs_d, 
                                         matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b, 
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, 
                                        strainPassCats=strainPassCats, labels=labels)
        
        test_prediction_ls = test_prediction_ls + output.view(-1).tolist()
        test_reference_ls = test_reference_ls + labels.tolist()
        test_logits_ls.append(logits.tolist())

test_mae = mean_absolute_error(test_reference_ls, test_prediction_ls)
test_mse = mean_squared_error(test_reference_ls, test_prediction_ls)
test_pearson = pearsonr(test_reference_ls, test_prediction_ls)
test_spearman = spearmanr(test_reference_ls, test_prediction_ls)
R2_score = r2_score(test_reference_ls, test_prediction_ls)

print(f"Test MAE: {test_mae}")
print(f"Test MSE: {test_mse}")
print(f"Test Pearson: {test_pearson}")
print(f"Test Spearman: {test_spearman}")
print(f"Test R2_score: {R2_score}")

Test MAE: 0.5691747341473592
Test MSE: 0.5304368335141715
Test Pearson: PearsonRResult(statistic=0.9122041062655805, pvalue=0.0)
Test Spearman: SignificanceResult(statistic=0.8127409815525211, pvalue=0.0)
Test R2_score: 0.8271510180997891


In [10]:
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr


test_prediction_ls = []
test_reference_ls = []
test_logits_ls = []
model.eval()
for batch in H3N2_dataloader:
    with torch.no_grad():
        emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch
        
        matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
        matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
        matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
        matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])
        
        matrixs_a = matrixs_a.to(device)
        matrixs_b = matrixs_b.to(device)
        matrixs_c = matrixs_c.to(device)
        matrixs_d = matrixs_d.to(device)

        masks_a = masks_a.to(device)
        masks_b = masks_b.to(device)
        masks_c = masks_c.to(device)
        masks_d = masks_d.to(device)
        
        strainPassCats = strainPassCats.to(device)
        
        labels = labels.to(device)
        
        with torch.no_grad():
            loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c, matrices_d=matrixs_d, 
                                         matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b, 
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, 
                                        strainPassCats=strainPassCats, labels=labels)
        
        test_prediction_ls = test_prediction_ls + output.view(-1).tolist()
        test_reference_ls = test_reference_ls + labels.tolist()
        test_logits_ls.append(logits.tolist())

test_mae = mean_absolute_error(test_reference_ls, test_prediction_ls)
test_mse = mean_squared_error(test_reference_ls, test_prediction_ls)
test_pearson = pearsonr(test_reference_ls, test_prediction_ls)
test_spearman = spearmanr(test_reference_ls, test_prediction_ls)
R2_score = r2_score(test_reference_ls, test_prediction_ls)

print(f"Test MAE: {test_mae}")
print(f"Test MSE: {test_mse}")
print(f"Test Pearson: {test_pearson}")
print(f"Test Spearman: {test_spearman}")
print(f"Test R2_score: {R2_score}")

Test MAE: 0.6016103416828804
Test MSE: 0.6228098946775278
Test Pearson: PearsonRResult(statistic=0.8973730898199043, pvalue=0.0)
Test Spearman: SignificanceResult(statistic=0.8994813799664002, pvalue=0.0)
Test R2_score: 0.8051310769463718


In [ ]:
import torch
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr


test_prediction_ls = []
test_reference_ls = []
test_logits_ls = []
model.eval()
for batch in Crick_test_dataloader:
    with torch.no_grad():
        emb_file_name_a, emb_file_name_b, emb_file_name_c, emb_file_name_d, strainPassCats, labels = batch
        
        matrixs_a, masks_a = generate_matrix([emb_dict[key] for key in emb_file_name_a])
        matrixs_b, masks_b = generate_matrix([emb_dict[key] for key in emb_file_name_b])
        matrixs_c, masks_c = generate_matrix([emb_dict[key] for key in emb_file_name_c])
        matrixs_d, masks_d = generate_matrix([emb_dict[key] for key in emb_file_name_d])
        
        matrixs_a = matrixs_a.to(device)
        matrixs_b = matrixs_b.to(device)
        matrixs_c = matrixs_c.to(device)
        matrixs_d = matrixs_d.to(device)

        masks_a = masks_a.to(device)
        masks_b = masks_b.to(device)
        masks_c = masks_c.to(device)
        masks_d = masks_d.to(device)
        
        strainPassCats = strainPassCats.to(device)
        
        labels = labels.to(device)
        
        with torch.no_grad():
            loss, logits, output = model(matrices_a=matrixs_a, matrices_b=matrixs_b, matrices_c=matrixs_c, matrices_d=matrixs_d, 
                                         matrix_attention_masks_a=masks_a, matrix_attention_masks_b=masks_b, 
                                        matrix_attention_masks_c=masks_c, matrix_attention_masks_d=masks_d, 
                                        strainPassCats=strainPassCats, labels=labels)
        
        test_prediction_ls = test_prediction_ls + output.view(-1).tolist()
        test_reference_ls = test_reference_ls + labels.tolist()
        test_logits_ls.append(logits.tolist())

test_mae = mean_absolute_error(test_reference_ls, test_prediction_ls)
test_mse = mean_squared_error(test_reference_ls, test_prediction_ls)
test_pearson = pearsonr(test_reference_ls, test_prediction_ls)
test_spearman = spearmanr(test_reference_ls, test_prediction_ls)
R2_score = r2_score(test_reference_ls, test_prediction_ls)

print(f"Test MAE: {test_mae}")
print(f"Test MSE: {test_mse}")
print(f"Test Pearson: {test_pearson}")
print(f"Test Spearman: {test_spearman}")
print(f"Test R2_score: {R2_score}")

Test MAE: 0.5844512966411576
Test MSE: 0.5739428107097525
Test Pearson: PearsonRResult(statistic=0.9219001388801346, pvalue=0.0)
Test Spearman: SignificanceResult(statistic=0.9039271984439847, pvalue=0.0)
Test R2_score: 0.8486358136352787


: 